In [597]:
import numpy as np

In [598]:
UP, RIGHT, DOWN, LEFT = 0, 1, 2, 3
DIRS = {UP: (-1, 0), RIGHT: (0, 1), DOWN: (1, 0), LEFT: (0, -1)}
ARROWS = {UP: "↑", RIGHT: "→", DOWN: "↓", LEFT: "←"}

In [599]:
class GridWorld:

    def __init__(
        self,
        rows: int,
        cols: int,
        step_reward: float,
        terminals: dict[tuple[int, int], float],
        walls: set[tuple[int, int]],
    ):
        self.rows = rows
        self.cols = cols
        self.step_reward = step_reward
        self.terminals = terminals
        self.walls = walls
        self.s2c = [
            (r, c)
            for r in range(self.rows)
            for c in range(self.cols)
            if (r, c) not in self.walls
        ]
        self.nS = len(self.s2c)
        self.nA = 4
        self.c2s = {cell: i for i, cell in enumerate(self.s2c)}

    def step(self, s: int, a: int):
        dr, dc = DIRS[a]
        cell = self.s2c[s]
        nxt = (cell[0] + dr, cell[1] + dc)
        if (
            not 0 <= nxt[0] < self.rows
            or not 0 <= nxt[1] < self.cols
            or nxt in self.walls
        ):
            nxt = cell
        reward = self.terminals.get(nxt, self.step_reward)
        return self.c2s[nxt], reward
    
    def cell_repr(self, r, c):
        cell = (r, c)
        if cell in self.terminals:
            return f'{self.terminals[cell]:+}'
        elif cell in self.walls:
            return '#'
        else:
            return '·'
        
    def render(self):
        for r in range(self.rows):
            for c in range(self.cols):
                if r == 0 and c == 0:
                    print(" r/c", end="")
                    print(''.join([f'{v:>3} ' for v in range(self.cols)]))
                if c == 0:
                    print(f'{r:>3} ', end="")
                print(f'{self.cell_repr(r, c):>3} ', end="")
            print()
            
    def __repr__(self):
        return f'''
Grid(
    rows={self.rows},
    cols={self.cols},
    step_reward={self.step_reward},
    terminals={self.terminals},
    walls={self.walls},
)
'''
env = GridWorld(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)
env


Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)

In [600]:
env.render()

 r/c  0   1   2   3 
  0   ·   ·   ·  +1 
  1   ·   #   ·  -1 
  2   ·   ·   ·   · 


### value iteration

In [601]:
def show_V(env: GridWorld, V: np.ndarray):
    for r in range(env.rows):
        for c in range(env.cols):
            if r == 0 and c == 0:
                print("  r/c", end="")
                print(''.join([f'{v:>4} ' for v in range(env.cols)]))
            if c == 0:
                print(f'{r:>4} ', end="")            
            cell = (r, c)
            if cell in env.c2s:
                value = round(V[env.c2s[(r, c)]], 2)
            else:
                value = '#'
            print(f'{value:>4} ', end="")
        print()

def q_from_v(
    env: GridWorld,
    V: np.ndarray,
    s: int,
    gamma: float,
):
    q = np.zeros(env.nA)
    if env.s2c[s] in env.terminals:
        return q
    for a in range(env.nA):
        ns, r = env.step(s, a)
        q[a] = r + gamma * V[ns]
    return q

def value_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
):
    V = np.zeros(env.nS)
    if verbose >= 2:
        print('---------- value_iteration ------------')
        print('V init')
        show_V(env, V)
        print('-' * 25)    
    delta = float('inf')
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = V.copy()
        for s in range(env.nS):
            if env.s2c[s] in env.terminals:
                continue
            V[s] = np.max(q_from_v(env, V_old, s, gamma))
            delta = max(delta, abs(V[s] - V_old[s]))
        if verbose >= 1:
            print(f"iter {i}: delta={delta:.6f}")
        if verbose >= 2:
            show_V(env, V)
            print('-' * 25)
        i += 1
    converged = delta < theta
    if verbose >= 1:
        if converged:
                print(f'value_iteration converged in {i - 1} iterations')
        else:
            print(f"value_iteration did not converge in {max_iters} iterations")
    return V, converged

In [602]:
V, converged = value_iteration(env)
V

array([0.81  , 0.9   , 1.    , 0.    , 0.729 , 0.9   , 0.    , 0.6561,
       0.729 , 0.81  , 0.729 ])

In [603]:
show_V(env, V)

  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2 0.66 0.73 0.81 0.73 


In [604]:
value_iteration(env, verbose=2);

---------- value_iteration ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=0.900000
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 2: delta=0.810000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0  0.0 0.81  0.0 
-------------------------
iter 3: delta=0.729000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
iter 4: delta=0.656100
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2 0.66 0.73 0.81 0.73 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0 0.81

In [605]:
def read_policy(
    env: GridWorld,
    V: np.ndarray,
    gamma=0.9,
):
    policy = np.full(env.nS, -1)
    for s in range(env.nS):
        if env.s2c[s] in env.terminals:
            continue
        best_a = int(np.argmax(q_from_v(env, V, s, gamma)))
        policy[s] = best_a
    return policy
policy = read_policy(env, V)
policy

array([ 1,  1,  1, -1,  0,  0, -1,  0,  1,  0,  3])

In [606]:
def render_policy(env: GridWorld, policy: np.ndarray):
    for r in range(env.rows):
        for c in range(env.cols):
            if r == 0 and c == 0:
                print(" r/c", end="")
                print(''.join([f'{v:>3} ' for v in range(env.cols)]))
            if c == 0:
                print(f'{r:>3} ', end="")  

            cell = (r, c)
            if cell in env.c2s:
                if cell in env.terminals:
                    value = f'{env.terminals[cell]:+}'
                else:
                    value = ARROWS[policy[env.c2s[(r, c)]]]
            else:
                value = '#'
            print(f'{value:>3} ', end='')
        print()    
render_policy(env, policy)

 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 


### read policy during value_iteration 

we can also read policy during value_iteration, no need to do it with returned V

In [607]:
def value_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
):
    V = np.zeros(env.nS)
    policy = np.full(env.nS, -1)
    if verbose >= 2:
        print('---------- value_iteration ------------')
        print('V init')
        show_V(env, V)
        print('-' * 25)    
    delta = float('inf')
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = V.copy()
        for s in range(env.nS):
            if env.s2c[s] in env.terminals:
                continue
            q = q_from_v(env, V_old, s, gamma)
            V[s] = np.max(q)
            policy[s] = int(np.argmax(q))
            delta = max(delta, abs(V[s] - V_old[s]))
        if verbose >= 1:
            print(f"iter {i}: delta={delta:.6f}")
        if verbose >= 2:
            show_V(env, V)
            print('-' * 25)
        i += 1
    converged = delta < theta
    if verbose >= 1:
        if converged:
                print(f'value_iteration converged in {i - 1} iterations')
        else:
            print(f"value_iteration did not converge in {max_iters} iterations")
    return policy, V, converged

In [608]:
policy, V, converged = value_iteration(env)
policy

array([ 1,  1,  1, -1,  0,  0, -1,  0,  1,  0,  3])

In [609]:
render_policy(env, policy)

 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 


### policy iteration

In [610]:
def policy_evaluation(
    env: GridWorld,
    policy: np.ndarray,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
):
    V = np.zeros(env.nS)
    if verbose >= 2:
        print('---------- policy_evaluation ------------')
        print('V init')
        show_V(env, V)
        print('-' * 25)    
    delta = float('inf')
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = V.copy()
        for s in range(env.nS):
            if env.s2c[s] in env.terminals:
                continue
            a = policy[s]
            ns, r = env.step(s, a)
            V[s] = r + gamma * V_old[ns]
            delta = max(delta, abs(V[s] - V_old[s]))
        if verbose >= 1:
            print(f"iter {i}: delta={delta:.6f}")
        if verbose >= 2:
            show_V(env, V)
            print('-' * 25)
        i += 1
    converged = delta < theta
    if verbose >= 1:
        if converged:
            print(f'policy_evaluation converged in {i - 1} iterations')
        else:
            print(f"policy_evaluation did not converge in {max_iters} iterations")
    return V, converged

In [611]:
V, converged = policy_evaluation(env, policy, verbose=2)

---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=0.900000
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 2: delta=0.810000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0  0.0 0.81  0.0 
-------------------------
iter 3: delta=0.729000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
iter 4: delta=0.656100
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2 0.66 0.73 0.81 0.73 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0 0.

In [612]:
def policy_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
    pe_max_iters=1000,
    pe_verbose=0,
):
    policy = np.full(env.nS, UP)
    for cell in env.terminals:
        policy[env.c2s[cell]] = -1
    V = np.zeros(env.nS)

    if verbose >= 2:
        print('---------- policy_iteration ------------')
        print('policy init')
        render_policy(env, policy)
    
    i = 0
    pe_converged = True
    changed = None
    while changed != 0 and i < max_iters:
        V, pe_converged = policy_evaluation(
            env, policy, gamma, theta, max_iters=pe_max_iters, verbose=pe_verbose
        )
        if not pe_converged:
            break
        new_policy = read_policy(env, V, gamma)
        changed = int(np.count_nonzero(new_policy != policy))
        if verbose >= 2:
            print('-' * 25)  
        if verbose >= 1:
            print(f'iter {i}: actions changed = {changed}')
        if verbose >= 2:
            render_policy(env, new_policy)
        policy = new_policy
        i += 1
    converged = changed == 0
    if verbose >= 1:
        print('-' * 25)  
        if not pe_converged:
            print(
                f"policy_iteration did not converge because policy_evaluation did not converge"
            )
        elif converged:
            print(f'policy_iteration converged in {i - 1} iterations')
        else:
            print(f'policy_iteration did not converge in {max_iters} iterations')
    return policy, V, converged    

In [613]:
policy, V, converged = policy_iteration(env, verbose=2)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ↑ 
-------------------------
iter 0: actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
-------------------------
iter 1: actions changed = 2
 r/c  0   1   2   3 
  0   ↑   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 
-------------------------
iter 2: actions changed = 2
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   →   →   ↑   ← 
-------------------------
iter 3: actions changed = 1
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 
-------------------------
iter 4: actions changed = 0
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 
-------------------------
policy_iteration converged in 4 iterations


In [614]:
policy, V, converged = policy_iteration(env, verbose=2, pe_verbose=2)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ↑ 
---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
iter 1: delta=0.000000
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
policy_evaluation converged in 1 iterations
-------------------------
iter 0: actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.0

### don't update action until better is available

currently tied action at cell (2, 0), get's flipped from RIGHT to UP, we can make it so that action isn't updated until strictly better is available

In [615]:
def read_policy(
    env: GridWorld,
    V: np.ndarray,
    gamma=0.9,
    theta=1e-6,
    incumbent_policy: np.ndarray = None,
):
    policy = np.full(env.nS, -1)
    for s in range(env.nS):
        if env.s2c[s] in env.terminals:
            continue
        q = q_from_v(env, V, s, gamma)
        new_a = int(np.argmax(q))
        if incumbent_policy is not None:
            incumbent_a = incumbent_policy[s]
            if incumbent_a != -1 and q[new_a] - q[incumbent_a] < 10 * theta:
                new_a = incumbent_a
        policy[s] = new_a
    return policy

In [616]:
def policy_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    pass_incumbent_policy=False,
    verbose=0,
    pe_max_iters=1000,
    pe_verbose=0,
):
    policy = np.full(env.nS, UP)
    for cell in env.terminals:
        policy[env.c2s[cell]] = -1
    V = np.zeros(env.nS)

    if verbose >= 2:
        print('---------- policy_iteration ------------')
        print('policy init')
        render_policy(env, policy)

    i = 0
    pe_converged = True
    changed = None
    while changed != 0 and i < max_iters:
        V, pe_converged = policy_evaluation(
            env, policy, gamma, theta, max_iters=pe_max_iters, verbose=pe_verbose
        )
        if not pe_converged:
            break
        new_policy = read_policy(
            env,
            V,
            gamma,
            theta,
            incumbent_policy=policy if pass_incumbent_policy else None,
        )
        changed = int(np.count_nonzero(new_policy != policy))
        if verbose >= 2:
            print('-' * 25)  
        if verbose >= 1:
            print(f'iter {i}: actions changed = {changed}')
        if verbose >= 2:
            render_policy(env, new_policy)
        policy = new_policy
        i += 1
    converged = changed == 0
    if verbose >= 1:
        print('-' * 25)  
        if not pe_converged:
            print(
                f"policy_iteration did not converge because policy_evaluation did not converge"
            )
        elif converged:
            print(f'policy_iteration converged in {i - 1} iterations')
        else:
            print(f'policy_iteration did not converge in {max_iters} iterations')
    return policy, V, converged    

In [617]:
policy, V, converged = policy_iteration(env, pass_incumbent_policy=True, verbose=2)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ↑ 
-------------------------
iter 0: actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
-------------------------
iter 1: actions changed = 2
 r/c  0   1   2   3 
  0   ↑   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 
-------------------------
iter 2: actions changed = 2
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   →   →   ↑   ← 
-------------------------
iter 3: actions changed = 0
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   →   →   ↑   ← 
-------------------------
policy_iteration converged in 3 iterations


### pass previous V to policy_evaluation

policy_evaluation throws away its previous V. Every call restarts from np.zeros, which is why the pe_verbose=2 trace shows 4–5 sweeps each time. Warm-starting from the previous evaluation is the standard speedup — the new policy's V is close to the old one's, so it usually converges in 1–2 sweeps.

In [618]:
def policy_evaluation(
    env: GridWorld,
    policy: np.ndarray,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
    V: np.ndarray = None
):
    V = np.zeros(env.nS) if V is None else V.copy()
    if verbose >= 2:
        print('---------- policy_evaluation ------------')
        print('V init')
        show_V(env, V)
        print('-' * 25)    
    delta = float('inf')
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = V.copy()
        for s in range(env.nS):
            if env.s2c[s] in env.terminals:
                continue
            a = policy[s]
            ns, r = env.step(s, a)
            V[s] = r + gamma * V_old[ns]
            delta = max(delta, abs(V[s] - V_old[s]))
        if verbose >= 1:
            print(f"iter {i}: delta={delta:.6f}")
        if verbose >= 2:
            show_V(env, V)
            print('-' * 25)
        i += 1
    converged = delta < theta
    if verbose >= 1:
        if converged:
            print(f'policy_evaluation converged in {i - 1} iterations')
        else:
            print(f"policy_evaluation did not converge in {max_iters} iterations")
    return V, converged

In [619]:
def policy_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    pass_incumbent_policy=False,
    verbose=0,
    pe_max_iters=1000,
    pe_pass_prev_V=False,
    pe_verbose=0,
):
    policy = np.full(env.nS, UP)
    for cell in env.terminals:
        policy[env.c2s[cell]] = -1
    V = np.zeros(env.nS)

    if verbose >= 2:
        print('---------- policy_iteration ------------')
        print('policy init')
        render_policy(env, policy)

    i = 0
    pe_converged = True
    changed = None
    while changed != 0 and i < max_iters:
        V, pe_converged = policy_evaluation(
            env,
            policy,
            gamma,
            theta,
            max_iters=pe_max_iters,
            verbose=pe_verbose,
            V=V if pe_pass_prev_V else None,
        )
        if not pe_converged:
            break
        new_policy = read_policy(
            env,
            V,
            gamma,
            theta,
            incumbent_policy=policy if pass_incumbent_policy else None,
        )
        changed = int(np.count_nonzero(new_policy != policy))
        if verbose >= 2:
            print('-' * 25)  
        if verbose >= 1:
            print(f'iter {i}: actions changed = {changed}')
        if verbose >= 2:
            render_policy(env, new_policy)
        policy = new_policy
        i += 1
    converged = changed == 0
    if verbose >= 1:
        print('-' * 25)  
        if not pe_converged:
            print(
                f"policy_iteration did not converge because policy_evaluation did not converge"
            )
        elif converged:
            print(f'policy_iteration converged in {i - 1} iterations')
        else:
            print(f'policy_iteration did not converge in {max_iters} iterations')
    return policy, V, converged    

In [620]:
policy, V, converged = policy_iteration(env, verbose=2, pe_verbose=2, pe_pass_prev_V=True)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ↑ 
---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
iter 1: delta=0.000000
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
policy_evaluation converged in 1 iterations
-------------------------
iter 0: actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
iter 0: delta=1.0

In [621]:
policy, V, converged = policy_iteration(
    env, verbose=2, pe_verbose=2, pe_pass_prev_V=True, pass_incumbent_policy=True
)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ↑ 
---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
iter 1: delta=0.000000
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
policy_evaluation converged in 1 iterations
-------------------------
iter 0: actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
iter 0: delta=1.0